# Student Dropout & Academic Success: 5-Model Classification
ML Assignment 2. Dataset: UCI *Predict Students' Dropout and Academic Success* (4,424 rows, 36 features, 3 classes).



In [ ]:

import json, os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.metrics import confusion_matrix, classification_report

In [ ]:
# metric helper (also saved as metrics_utils.py, which the Streamlit app imports)
METRICS_CODE = """import numpy as np
from sklearn.metrics import (accuracy_score, roc_auc_score, precision_score,
                             recall_score, f1_score, matthews_corrcoef)

def compute_metrics(y_true, y_pred, y_proba):
    # works for binary and multi-class (weighted averages)
    n_classes = y_proba.shape[1]
    if n_classes == 2:
        auc = roc_auc_score(y_true, y_proba[:, 1]); avg = "binary"
    else:
        auc = roc_auc_score(y_true, y_proba, multi_class="ovr", average="weighted"); avg = "weighted"
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "AUC": auc,
        "Precision": precision_score(y_true, y_pred, average=avg, zero_division=0),
        "Recall": recall_score(y_true, y_pred, average=avg, zero_division=0),
        "F1": f1_score(y_true, y_pred, average=avg, zero_division=0),
        "MCC": matthews_corrcoef(y_true, y_pred),
    }
"""
with open("metrics_utils.py", "w") as f:
    f.write(METRICS_CODE)

from metrics_utils import compute_metrics

## 1. Configuration and data loading

In [ ]:
# config
DATA_PATH = "/home/s2026ak05119/workspace/data.csv"
TARGET = "Target"
DROP_COLS = []
# integer-coded columns that are really categories (one-hot encoded instead of scaled)
CODED_CATS = ["Marital status", "Application mode", "Course", "Previous qualification",
              "Nacionality", "Mother's qualification", "Father's qualification",
              "Mother's occupation", "Father's occupation"]
TEST_SIZE = 0.2
SEED = 42

In [ ]:
#  load data 
with open(DATA_PATH, encoding="utf-8-sig") as f:
    first_line = f.readline()
sep = ";" if first_line.count(";") > first_line.count(",") else ","
df = pd.read_csv(DATA_PATH, sep=sep)
df.columns = df.columns.str.strip()

if TARGET not in df.columns:
    raise SystemExit(f"Target column '{TARGET}' not found. Columns are: {df.columns.tolist()}")

df = df.drop(columns=DROP_COLS).dropna(subset=[TARGET])
print("Shape:", df.shape)
print(df[TARGET].value_counts())
df.head()

In [ ]:
#  class distribution plot
ax = df[TARGET].value_counts().plot(kind="bar", color=["#4c78a8", "#f58518", "#54a24b"])
ax.set_title("Class distribution"); ax.set_ylabel("Students")
plt.tight_layout(); plt.show()

## 2. Preprocessing and train/test split

In [ ]:
# features, target, column groups
X, y_raw = df.drop(columns=[TARGET]), df[TARGET]

le = LabelEncoder()
y = le.fit_transform(y_raw)
print("Classes:", list(le.classes_))

num_cols = X.select_dtypes(include=np.number).columns.tolist()
coded = [c for c in CODED_CATS if c in X.columns]
num_cols = [c for c in num_cols if c not in coded]
cat_cols = [c for c in X.columns if c not in num_cols]
print(len(coded), "coded categorical columns found (expect 9)")
print(len(num_cols), "numeric columns,", len(cat_cols), "categorical columns")

def make_pre(scale=True):
    num_steps = [("imp", SimpleImputer(strategy="median"))]
    if scale:
        num_steps.append(("sc", StandardScaler()))
    return ColumnTransformer([
        ("num", Pipeline(num_steps), num_cols),
        ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                          ("oh", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), cat_cols),
    ])

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=TEST_SIZE, stratify=y, random_state=SEED)
print("Train:", X_tr.shape, "Test:", X_te.shape)

## 3. Train the five models

In [ ]:
# model definitions
models = {
    "Logistic Regression": Pipeline([("pre", make_pre()), ("clf", LogisticRegression(max_iter=2000))]),
    "Decision Tree": Pipeline([("pre", make_pre(False)), ("clf", DecisionTreeClassifier(max_depth=8, random_state=SEED))]),
    "KNN": Pipeline([("pre", make_pre()), ("clf", KNeighborsClassifier(n_neighbors=7))]),
    "Naive Bayes": Pipeline([("pre", make_pre()), ("clf", GaussianNB(var_smoothing=0.1))]),
    "Support Vector Machine": Pipeline([("pre", make_pre()), ("clf", SVC(kernel="rbf", probability=True, random_state=SEED))]),
}

In [ ]:
#  train, evaluate, save everything the Streamlit app needs
os.makedirs("model", exist_ok=True)

# test_data.csv = raw features + ORIGINAL target labels (this is the file you upload in the app)
test_df = X_te.copy()
test_df[TARGET] = le.inverse_transform(y_te)
test_df.to_csv("test_data.csv", index=False)

rows = []
for name, pipe in models.items():
    pipe.fit(X_tr, y_tr)
    joblib.dump(pipe, f"model/{name.replace(' ', '_').lower()}.joblib")
    m = compute_metrics(y_te, pipe.predict(X_te), pipe.predict_proba(X_te))
    rows.append({"ML Model Name": name, **m})
    print("done:", name)

joblib.dump(le, "model/label_encoder.joblib")
with open("model/meta.json", "w") as f:
    json.dump({"target": TARGET, "features": X.columns.tolist(), "classes": [str(c) for c in le.classes_]}, f)

results = pd.DataFrame(rows).round(4)
results.to_csv("model/metrics_comparison.csv", index=False)
print(results.to_markdown(index=False))

## 4. Confusion matrices and classification reports

In [ ]:
# Cell 10: confusion matrices for all five models
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for ax, (name, pipe) in zip(axes.ravel(), models.items()):
    cm = confusion_matrix(y_te, pipe.predict(X_te))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
                xticklabels=le.classes_, yticklabels=le.classes_)
    ax.set_title(name); ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
axes.ravel()[-1].axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# classification report for one model 
best = "Logistic Regression"
print(best)
print(classification_report(y_te, models[best].predict(X_te), target_names=le.classes_))

##  how Naive Bayes was tuned
With default settings GaussianNB scored only about 0.24 accuracy because the many one-hot columns have near-zero variance. `var_smoothing` was tuned with 5-fold cross-validation on the **training set only** (no test-set leakage), comparing GaussianNB with MultinomialNB. GaussianNB with `var_smoothing=0.1` scored best, so it is used . This cell reproduces the search.

In [ ]:
#  Naive Bayes tuning (cross-validation on training data only)
from sklearn.preprocessing import MinMaxScaler
from sklearn.naive_bayes import MultinomialNB

def pre_mn():
    return ColumnTransformer([
        ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("mm", MinMaxScaler())]), num_cols),
        ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                          ("oh", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), cat_cols),
    ])

candidates = {
    "Gaussian": (Pipeline([("pre", make_pre()), ("clf", GaussianNB())]),
                 {"clf__var_smoothing": [1e-9, 1e-6, 1e-3, 1e-2, 1e-1, 0.5, 1.0]}),
    "Multinomial": (Pipeline([("pre", pre_mn()), ("clf", MultinomialNB())]),
                    {"clf__alpha": [0.1, 0.5, 1.0, 2.0]}),
}
for name, (pipe, grid) in candidates.items():
    gs = GridSearchCV(pipe, grid, scoring="f1_weighted", cv=5, n_jobs=-1)
    gs.fit(X_tr, y_tr)
    print(name, gs.best_params_, round(gs.best_score_, 4))

## Versions used (For requirements.txt)

In [ ]:
import sys, numpy, scipy, sklearn, pandas, joblib, matplotlib, seaborn
print("python", sys.version.split()[0])
for m in [numpy, scipy, sklearn, pandas, joblib, matplotlib, seaborn]:
    print(m.__name__, m.__version__)